# Importing libraries

In [113]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore") 

# Data preprocessing

## Retrieve data

In [114]:
df = pd.read_csv('data/client_loan_data_processed.csv')

In [115]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 395067 entries, 0 to 395066
Data columns (total 33 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   arrearsDays                   32686 non-null   float64
 1   employmentStatus              81902 non-null   str    
 2   income                        55703 non-null   float64
 3   SalaryMonthCount              55703 non-null   float64
 4   HasConsistentSalaryAmount     55703 non-null   str    
 5   clientTenureDays              395061 non-null  float64
 6   loanCycleNumber               394065 non-null  float64
 7   loanCycleDeviation            37052 non-null   float64
 8   arrearsConsistencyAccount     33320 non-null   object 
 9   arrearsConsistencyInstalment  33317 non-null   object 
 10  netInflow                     312298 non-null  str    
 11  avgDailyBalance               394951 non-null  str    
 12  riskTier                      384389 non-null  str    


## Convert numeric values to categorical

Format arrears days

In [116]:
nonzero = df[df['arrearsDays'] > 0]['arrearsDays']
q33, q66, q95 = nonzero.quantile([0.33, 0.66, 0.95])

def categorize_arrears(days):
    if pd.isna(days) or days == 0.0:
        return 'No Risk'
    elif days <= q33:
        return 'Low Risk'
    elif days <= q66:
        return 'Medium Risk'
    elif days <= q95:
        return 'High Risk'
    else:
        return 'Critical Risk'

df['arrearsDaysCategory'] = df['arrearsDays'].apply(categorize_arrears)

Format income

In [117]:
def categorize_salary(income):
    if pd.isna(income) or income == 0:
        return 'Salary Not Recorded'
    else:
        return 'Salary Recorded'

df['salaryRecorded'] = df['income'].apply(categorize_salary)

In [118]:
df['salaryRecorded'].value_counts()

salaryRecorded
Salary Not Recorded    394950
Salary Recorded           117
Name: count, dtype: int64

Format client tenure

In [119]:
df = df[df['clientTenureDays']!= -24]

In [120]:
df['clientTenureDays'].describe()

count    395060.000000
mean        363.928272
std         453.342309
min           0.000000
25%          79.000000
50%         240.000000
75%         451.000000
max        3637.000000
Name: clientTenureDays, dtype: float64

In [121]:
def categorize_tenure(days):
    if days <= 90:
        return 'New Client'          # 0-3 months
    elif days <= 365:
        return 'Developing Client'   # 3-12 months
    elif days <= 730:
        return 'Established Client'  # 1-2 years
    else:
        return 'Long-Term Client'     # 2+ years

df['tenureCategory'] = df['clientTenureDays'].apply(categorize_tenure)

In [122]:
df['tenureCategory'].value_counts()

tenureCategory
Developing Client     151610
New Client            111613
Established Client     83731
Long-Term Client       48112
Name: count, dtype: int64

Recent loan tenure

In [123]:
nonzero = df[df['RecentLoanTenure'] > 0]['RecentLoanTenure']
q33, q66, q95 = nonzero.quantile([0.33, 0.66, 0.95])

def categorize_arrears(days):
    if pd.isna(days) or days == 0.0:
        return 'No Risk'
    elif days <= q33:
        return 'Low Risk'
    elif days <= q66:
        return 'Medium Risk'
    elif days <= q95:
        return 'High Risk'
    else:
        return 'Critical Risk'

df['recentLoanTenureCategory'] = df['RecentLoanTenure'].apply(categorize_arrears)

In [124]:
df['recentLoanTenureCategory'].value_counts()

recentLoanTenureCategory
No Risk          390132
Low Risk           1877
High Risk          1426
Medium Risk        1395
Critical Risk       236
Name: count, dtype: int64

Loan cycle number

In [125]:
def categorize_loan_cycle(cycle):
    if pd.isna(cycle):
        return 'Never Borrowed'
    elif cycle == 0:
        return 'First Time Borrower'
    elif cycle <= 2:
        return 'Early Repeat Borrower'
    elif cycle <= 5:
        return 'Established Repeat Borrower'
    else:
        return 'Veteran Borrower'

df['loanCycleCategory'] = df['loanCycleNumber'].apply(categorize_loan_cycle)

In [126]:
df['loanCycleCategory'].value_counts()

loanCycleCategory
First Time Borrower            365770
Early Repeat Borrower           15115
Established Repeat Borrower      7577
Veteran Borrower                 5602
Never Borrowed                   1002
Name: count, dtype: int64

Drop unnecessary columns

In [127]:
df.drop(columns=['arrearsDays', 'income', 'clientTenureDays', 'RecentLoanTenure',
                 'loanCycleNumber', 'RecentLoanDaysLate'], inplace=True)

In [128]:
df.info()

<class 'pandas.DataFrame'>
Index: 395066 entries, 0 to 395066
Data columns (total 32 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   employmentStatus              81902 non-null   str    
 1   SalaryMonthCount              55702 non-null   float64
 2   HasConsistentSalaryAmount     55702 non-null   str    
 3   loanCycleDeviation            37052 non-null   float64
 4   arrearsConsistencyAccount     33320 non-null   object 
 5   arrearsConsistencyInstalment  33317 non-null   object 
 6   netInflow                     312297 non-null  str    
 7   avgDailyBalance               394950 non-null  str    
 8   riskTier                      384389 non-null  str    
 9   investmentBalance             394950 non-null  str    
 10  transactionRate               312297 non-null  float64
 11  historicalAvgBalance          394950 non-null  str    
 12  CurrentAdb                    69241 non-null   float64
 13  